# Hosting SmoLAgents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. 

We will focus on a SmoLAgents with Amazon Bedrock model example. SmoLAgents is Hugging Face's lightweight agent framework designed for efficient tool calling and minimal overhead.

### Tutorial Details

| Information         | Details                                                                      |
|:--------------------|:-----------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                               |
| Agent type          | Single                                                                       |
| Agentic Framework   | SmoLAgents (Hugging Face)                                                    |
| LLM model           | Anthropic Claude Sonnet 4                                                    |
| Tutorial components | Hosting agent on AgentCore Runtime. Using SmoLAgents and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                               |
| Example complexity  | Easy                                                                         |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                 |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will use a SmoLAgents agent using Amazon Bedrock models

In our example we will use a very simple agent with three tools: `get_weather`, `get_current_time`, and `calculate`. 

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models with native SmoLAgents integration
* Using SmoLAgents lightweight framework
* Custom tool integration
* Regional model ID support for EU and US deployments


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* SmoLAgents toolkit
* Docker running

In [ ]:
#!pip install -r requirements.txt

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.



In [ ]:
%%writefile smolagents_bedrock.py
from smolagents import CodeAgent, tool, AmazonBedrockServerModel
from datetime import datetime
import argparse
import json
import os

# Custom tools for the agent
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location.
    
    Args:
        location: The city or location to get weather for
        
    Returns:
        Weather information as a string
    """
    # Dummy implementation - in production, you'd call a real weather API
    weather_data = {
        "New York": "Sunny, 72°F",
        "London": "Cloudy, 15°C", 
        "Tokyo": "Rainy, 18°C",
        "Paris": "Partly cloudy, 20°C",
        "San Francisco": "Foggy, 65°F",
        "Miami": "Hot and humid, 85°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}. It's probably nice though!")

@tool
def get_current_time() -> str:
    """Get the current date and time.
    
    Returns:
        Current date and time as a formatted string
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

@tool
def calculate(expression: str) -> str:
    """Perform mathematical calculations safely.
    
    Args:
        expression: Mathematical expression to evaluate (e.g., "2 + 3 * 4")
        
    Returns:
        Result of the calculation as a string
    """
    try:
        # Safe evaluation of mathematical expressions
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error calculating {expression}: {str(e)}"

# Initialize the Amazon Bedrock model using SmoLAgents native integration
# Determine the region and set appropriate model ID prefix
aws_region = os.getenv("AWS_REGION", "us-east-1")

# Set model ID prefix based on region
if aws_region.startswith("eu-"):
    model_prefix = "eu"
elif aws_region.startswith("us-"):
    model_prefix = "us"
else:
    # Default to us for other regions
    model_prefix = "us"

model_id = f"{model_prefix}.anthropic.claude-sonnet-4-20250514-v1:0"

# Create Amazon Bedrock model with SmoLAgents native support
model = AmazonBedrockServerModel(
    model_id=model_id,
    client_kwargs={'region_name': aws_region}
)

# Create the SmoLAgents agent with tools (no system_prompt parameter)
agent = CodeAgent(
    tools=[get_weather, get_current_time, calculate],
    model=model
)

def smolagents_bedrock(payload):
    """
    Invoke the SmoLAgents agent with a payload
    
    Args:
        payload: Dictionary containing the user input
        
    Returns:
        Agent response as a string
    """
    user_input = payload.get("prompt", "")
    print(f"User input: {user_input}")
    
    try:
        # Run the agent with the user input
        response = agent.run(user_input)
        print(f"Agent response: {response}")
        return response
    except Exception as e:
        error_msg = f"Error processing request: {str(e)}"
        print(error_msg)
        return error_msg

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = smolagents_bedrock(json.loads(args.payload))
    print(response)

#### Invoking local agent

In [ ]:
!python smolagents_bedrock.py '{"prompt": "What is the weather in New York?"}'

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### SmoLAgents with Amazon Bedrock model
Let's start with our SmoLAgents using Amazon Bedrock model. Other examples with different frameworks and models are available in the parent directories

In [ ]:
%%writefile smolagents_bedrock.py
from smolagents import CodeAgent, tool, AmazonBedrockServerModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from datetime import datetime
import argparse
import json
import os

app = BedrockAgentCoreApp()

# Custom tools for the agent
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location.
    
    Args:
        location: The city or location to get weather for
        
    Returns:
        Weather information as a string
    """
    # Dummy implementation - in production, you'd call a real weather API
    weather_data = {
        "New York": "Sunny, 72°F",
        "London": "Cloudy, 15°C", 
        "Tokyo": "Rainy, 18°C",
        "Paris": "Partly cloudy, 20°C",
        "San Francisco": "Foggy, 65°F",
        "Miami": "Hot and humid, 85°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}. It's probably nice though!")

@tool
def get_current_time() -> str:
    """Get the current date and time.
    
    Returns:
        Current date and time as a formatted string
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

@tool
def calculate(expression: str) -> str:
    """Perform mathematical calculations safely.
    
    Args:
        expression: Mathematical expression to evaluate (e.g., "2 + 3 * 4")
        
    Returns:
        Result of the calculation as a string
    """
    try:
        # Safe evaluation of mathematical expressions
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error calculating {expression}: {str(e)}"

# Initialize the Amazon Bedrock model using SmoLAgents native integration
# Determine the region and set appropriate model ID prefix
aws_region = os.getenv("AWS_REGION", "us-east-1")

# Set model ID prefix based on region
if aws_region.startswith("eu-"):
    model_prefix = "eu"
elif aws_region.startswith("us-"):
    model_prefix = "us"
else:
    # Default to us for other regions
    model_prefix = "us"

model_id = f"{model_prefix}.anthropic.claude-sonnet-4-20250514-v1:0"

# Create Amazon Bedrock model with SmoLAgents native support
model = AmazonBedrockServerModel(
    model_id=model_id,
    client_kwargs={'region_name': aws_region}
)

# Create the SmoLAgents agent with tools (no system_prompt parameter)
agent = CodeAgent(
    tools=[get_weather, get_current_time, calculate],
    model=model
)

@app.entrypoint
def smolagents_bedrock(payload):
    """
    Invoke the SmoLAgents agent with a payload
    
    Args:
        payload: Dictionary containing the user input
        
    Returns:
        Agent response as a string
    """
    user_input = payload.get("prompt", "")
    print(f"User input: {user_input}")
    
    try:
        # Run the agent with the user input
        response = agent.run(user_input)
        print(f"Agent response: {response}")
        return response
    except Exception as e:
        error_msg = f"Error processing request: {str(e)}"
        print(error_msg)
        return error_msg

if __name__ == "__main__":
    app.run()

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## SmoLAgents Native Bedrock Integration

SmoLAgents provides native support for Amazon Bedrock through the `AmazonBedrockServerModel` class, which:

* Connects directly to Amazon Bedrock without additional dependencies
* Supports all Bedrock models including Claude, Nova, and others
* Handles authentication and region configuration automatically
* Provides optimal performance for Bedrock deployments
* Uses the correct SmoLAgents API (no `system_prompt` parameter for CodeAgent)
* Automatically selects the correct model ID prefix based on deployment region (us. or eu.)

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Creating runtime role

Before starting, let's create an IAM role for our AgentCore Runtime. We will do so using the utils function pre-developed for you.

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.join(utils_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role

agent_name="smolagents_bedrock"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="smolagents_bedrock.py",
    execution_role=agentcore_iam_role['Role']['Arn'],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

## Invoking your AgentCore Runtime

Now that your AgentCore Runtime is ready, let's invoke it using the starter toolkit. The starter toolkit provides a convenient `invoke` method that handles the communication with your deployed agent.

<div style="text-align:left">
    <img src="images/invoke.png" width="75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is the weather in New York?"})
invoke_response

### Processing invocation results

We can now process our invocation results to include it in an application

In [ ]:
from IPython.display import Markdown, display
import json
response_text = json.loads(invoke_response['response'][0].decode("utf-8"))
display(Markdown(response_text))

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

iam_client = boto3.client('iam')

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

policies = iam_client.list_role_policies(
    RoleName=agentcore_iam_role['Role']['RoleName'],
    MaxItems=100
)

for policy_name in policies['PolicyNames']:
    iam_client.delete_role_policy(
        RoleName=agentcore_iam_role['Role']['RoleName'],
        PolicyName=policy_name
    )
iam_response = iam_client.delete_role(
    RoleName=agentcore_iam_role['Role']['RoleName']
)

# Congratulations!

You have successfully:

* Created a SmoLAgents agent with Amazon Bedrock models
* Deployed the agent to Amazon Bedrock AgentCore Runtime
* Invoked the agent using both the starter toolkit and boto3
* Learned how to clean up resources

Your SmoLAgents agent is now ready for production use with automatic regional model selection!